# 💵 06 — Giá/km & Giá/tốc độ — hướng cải thiện model GIÁ

Đang tập trung cải thiện model **GIÁ** (surge đã ổn). Ý tưởng: thay vì model học trực tiếp
`target_shown_price` (biến động lớn, R²~0.66), thử học **đơn giá theo km** (`price_per_km`) —
nếu đơn giá này ổn định/dễ đoán hơn, ta có thể **đoán đơn giá rồi × quãng đường** để ra giá cuối,
avoid học trực tiếp 1 con số dao động lớn.

Dataset đã có sẵn cột `target_price_per_km` — xác nhận đúng bằng `target_shown_price / quote_distance`
(khớp 100%, không phải công thức khác).

In [ ]:
import sys; sys.path.insert(0, ".")
from _common import *
setup()
df = load(frac=0.15)
df["price_per_km"] = df[PRICE] / df.quote_distance.clip(lower=0.1)
print(f"Gia/km  median: {df.price_per_km.median():,.0f} VND/km | std: {df.price_per_km.std():,.0f}")
print(f"He so bien thien (CV) gia goc     : {(df[PRICE].std()/df[PRICE].mean()):.3f}")
print(f"He so bien thien (CV) gia/km      : {(df.price_per_km.std()/df.price_per_km.mean()):.3f}")
print("-> CV thap hon = gia/km ON DINH hon gia goc -> co the de doan hon.")

**1. Giá/km có giảm dần theo quãng đường không? (kinh tế theo quy mô)**

Taxi/ride-hailing thường có cấu trúc: **phí mở cửa cố định + phí/km** → chuyến càng dài, phí mở cửa
càng "loãng" ra, nên đơn giá/km càng **giảm dần**. Kiểm tra xem HCM có đúng mô hình này không.

In [ ]:
g = df.groupby(binned(df.quote_distance, 10)).price_per_km.median()
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(range(len(g)), g.values, color=BLUE)
ax.set_xticks(range(len(g))); ax.set_xticklabels(g.index, rotation=30, fontsize=8, ha="right")
ax.set_ylabel("Gia/km median (VND)"); ax.set_title("Gia/km giam dan theo quang duong?", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"corr(quang duong, gia/km) = {df.quote_distance.corr(df.price_per_km):.3f}")

**2. Giá/km có tăng khi tắc đường không? (tốc độ / dur_per_km)**

Nếu cước tính thêm theo phút di chuyển, chuyến tắc (tốc độ thấp) sẽ có đơn giá/km cao hơn cho cùng
quãng đường — đã thấy ở `05_kmpertime.ipynb` với giá tổng, giờ xem cụ thể trên đơn giá/km.

In [ ]:
corr_speed = df.speed_kmh.corr(df.price_per_km)
corr_durkm = df.dur_per_km.corr(df.price_per_km)
print(f"corr(toc do km/h, gia/km) = {corr_speed:.3f}  (am = toc do thap -> gia/km cao, dung ky vong)")
print(f"corr(phut/km, gia/km)     = {corr_durkm:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13,4))
g1 = df.groupby(binned(df.dur_per_km, 8)).price_per_km.median()
ax[0].bar(range(len(g1)), g1.values, color=RED)
ax[0].set_xticks(range(len(g1))); ax[0].set_xticklabels(g1.index, rotation=30, fontsize=8, ha="right")
ax[0].set_title("Gia/km theo muc tac (phut/km)", fontweight="bold"); ax[0].set_ylabel("Gia/km (VND)")

s = df.sample(min(8000,len(df)), random_state=0)
ax[1].scatter(s.speed_kmh.clip(0,60), s.price_per_km.clip(0, s.price_per_km.quantile(.99)), s=5, alpha=.15, color=BLUE)
ax[1].set_xlabel("Toc do (km/h)"); ax[1].set_ylabel("Gia/km (VND)"); ax[1].set_title("Gia/km vs toc do", fontweight="bold")
plt.tight_layout(); plt.show()

**3. Giá/tốc độ (`price_per_km` chuẩn hóa theo dịch vụ + giờ)**

Xem đơn giá/km trung bình theo **loại xe** và **giờ trong ngày** — để biết biến động đơn giá có
theo pattern rõ ràng (dịch vụ, giờ cao điểm) hay ngẫu nhiên.

In [ ]:
g_service = df.groupby("service_name").price_per_km.median()
g_hour = df.groupby("gio_vn").price_per_km.median()
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].bar(g_service.index, g_service.values, color=[BLUE,ORANGE])
ax[0].set_title("Gia/km median theo dich vu", fontweight="bold"); ax[0].set_ylabel("VND/km")
ax[1].plot(g_hour.index, g_hour.values, marker="o", color=GREEN, lw=2)
ax[1].set_xticks(range(0,24,2)); ax[1].set_xlabel("gio VN"); ax[1].set_ylabel("VND/km")
ax[1].set_title("Gia/km theo gio trong ngay", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"eta(dich vu, gia/km) = {eta(df.service_name, df.price_per_km):.3f}")
print(f"eta(gio, gia/km)     = {eta(df.gio_vn, df.price_per_km):.3f}")

**4. Kiểm chứng — dự đoán GIÁ/KM rồi × quãng đường có tốt hơn dự đoán GIÁ thẳng không?**

So 2 cách trên cùng feature (distance, duration, service, giờ, lịch sử giá, giá quan sát gần nhất):
- **(A) Dự đoán thẳng** `target_shown_price` (log)
- **(C) Dự đoán `price_per_km`** (log), rồi nhân lại với `quote_distance` để ra giá cuối

Đánh giá cả 2 trên **cùng thước đo cuối cùng: giá VND thực tế** (không so sánh R² của price_per_km
với R² của giá — 2 target khác thang đo, so trực tiếp sẽ sai lệch).

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OrdinalEncoder

CAT = ["service_name","pickup_location_name","dropoff_location_name"]
NUM = ["quote_distance","quote_duration","gio_vn","latest_observed_price",
       "history_60m_price_mean","history_60m_price_std"]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X = df[NUM].copy()
X[CAT] = enc.fit_transform(df[CAT].astype(str))
X = X[CAT+NUM]

idx_tr, idx_te = train_test_split(df.index, test_size=0.2, random_state=42)
Xtr, Xte = X.loc[idx_tr], X.loc[idx_te]
dist_te = df.loc[idx_te, "quote_distance"].clip(lower=0.1)
price_te = df.loc[idx_te, PRICE]

def tao():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.06, categorical_features=CAT, random_state=42)

# (A) Du doan thang gia
mA = tao().fit(Xtr, np.log(df.loc[idx_tr, PRICE]))
predA = np.exp(mA.predict(Xte))

# (C) Du doan gia/km roi nhan lai quang duong
mC = tao().fit(Xtr, np.log(df.loc[idx_tr, "price_per_km"]))
predC_perkm = np.exp(mC.predict(Xte))
predC = predC_perkm * dist_te

print("(A) Du doan thang GIA:")
print(f"    R2={r2_score(price_te,predA):.4f}  MAE={mean_absolute_error(price_te,predA):,.0f} VND  MAPE={np.mean(np.abs((price_te-predA)/price_te))*100:.1f}%")
print("(C) Du doan GIA/KM roi nhan lai quang duong:")
print(f"    R2={r2_score(price_te,predC):.4f}  MAE={mean_absolute_error(price_te,predC):,.0f} VND  MAPE={np.mean(np.abs((price_te-predC)/price_te))*100:.1f}%")
print(f"\nChenh lech MAE (C - A) = {mean_absolute_error(price_te,predC)-mean_absolute_error(price_te,predA):+,.0f} VND")
print("-> Am (C tot hon): dang chuyen model sang huong nay.")
print("-> Duong/gan 0: khong co loi, giu nguyen cach du doan thang gia.")

**Kết luận**

- Bước 1-3: nếu `price_per_km` biến động **ít hơn** giá gốc (CV thấp hơn — xem bước 0) và có pattern
  rõ theo quãng đường/tắc đường/dịch vụ/giờ → đây là tín hiệu tốt cho việc đổi hướng target.
- Bước 4 là câu trả lời thực nghiệm trực tiếp: so MAE/MAPE trên **cùng đơn vị VND cuối cùng** giữa
  2 cách. Nếu (C) thắng rõ → nên đổi `train_hybrid.ipynb`/`train_gia.ipynb` sang học `price_per_km`
  thay vì `target_shown_price`/`base_price` trực tiếp — cùng ý tưởng phân rã như Hybrid (base×surge)
  nhưng ở một trục khác (giá = đơn giá/km × quãng đường).